
# CRIM Intervals:  Melodic and Harmonic Corpus Search

### What You Can Do with this Notebook:

* Search A Corpus for Melodic and Harmonic nGrams

### A. Import Intervals and Other Code


In [1]:
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
import pandas as pd
import re
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact
from pandas import json_normalize
from pyvis.network import Network
from IPython.display import display
import requests
import os
import glob as glob


MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)
else:
    print(MUSDIR, "folder already exists.")

saved_csv folder already exists.
Music_Files folder already exists.


In [2]:
import types
import httpx

def verovioPrintExampleBig(self, start, stop):
    """
    Pass a range of measures (as integers) to print the given range.

    For last measure you can also use '-1', thus for all measures:

    verovioPrintExample(1, -1)

    """
    if self.path.startswith('Music_Files/'):
        text_file = open(self.path, "r")
        fetched_mei_string = text_file.read()
    elif self.path.startswith('/'):
        text_file = open(self.path, "r")
        fetched_mei_string = text_file.read()
    else:
        response = httpx.get(self.path)
        fetched_mei_string = response.text
    tk = verovio.toolkit()
    tk.loadData(fetched_mei_string)
    tk.setScale(30)
    tk.setOptions({"pageHeight":  3000, # Height in pixels
                    "pageWidth":  3000    # Width in pixels
                    })

    if stop == -1:
        meas = self.measures()
        stop = meas.iloc[-1].tolist()[0]

    mr = str(start) + "-" + str(stop)
    mdict = {'measureRange': mr}

    if stop < start:
        print("Check the measure range, the stop measure must be equal to or greater than the start measure")
    else:
        tk.select(mdict)
        tk.redoLayout()

        print("Score:")
        count = tk.getPageCount()
        for c in range(1, count + 1):
            music = tk.renderToSVG(c)
            print("File Name: ", self.file_name)
            print(self.metadata['composer'])
            print(self.metadata['title'])
            print("Measures: " + str(start) + "-" + str(stop))
            display(HTML(music))

In [3]:
corpus_list = sorted(glob.glob('Music_Files/Bona_*'))
corpus_list = [path.replace('\\', '/') for path in corpus_list]
corpus = CorpusBase(corpus_list)
corpus_list

['Music_Files/Bona_2.musicxml']

### Imposta i tuoi parametri

In [4]:
# numero di rapporti di valore
n = 4

## Versione non interattiva

In [5]:
list_ng_dfs = []

for url in corpus_list:
    # import to intervals
    piece = importScore(url)
    # get intervals metadata
    metadata = piece.metadata
    # get the path
    pathname = Path(url).name
    # add path to metadata
    metadata['path'] = pathname
    # run intervals function

    nr = piece.notes()
    dr = piece.durationalRatios().round(2).map(lambda x: str(x) if pd.notna(x) else x)
    ng = piece.ngrams(df=dr, n= 4)
    nv = piece.numberParts(df=ng)
    di = piece.detailIndex(df=nv, beat=False, offset=False) #to number parts, uncomment previous line and change df = to nv
    
    for key, value in metadata.items():
        di[key] = value
        
    list_ng_dfs.append(di)

output = pd.concat(list_ng_dfs)
    
# Assuming df is your DataFrame
new_order_check = ['title', 'composer', 'date', 'path']
new_order = ['title', 'composer', 'date', 'path'] + [col for col in output.columns if col not in new_order_check]
output = output[new_order].fillna('')
output = output.reset_index()
    
#define the function to convert tuples to strings
def convertTuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out  
    
#apply function to all cells
output.iloc[:, 5:] = output.iloc[:, 5:].map(convertTuple)
output

,Measure,title,composer,date,path,1,2,3,4,5,...,7,8,9,10,11,12,13,14,15,16
0,1.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,"0.33, 2.0, 1.0, 1.0",...,"0.33, 2.0, 1.0, 0.5","0.33, 2.0, 1.0, 1.0",,,,,,,,
1,1.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,"2.0, 1.0, 1.0, 1.0","2.0, 1.0, 1.0, 1.0","2.0, 1.0, 1.0, 1.0",,"2.0, 1.0, 1.0, 1.0",...,"2.0, 1.0, 0.5, 1.0","2.0, 1.0, 1.0, 1.0",,,,,,,,
2,1.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,"1.0, 1.0, 1.0, 1.5","1.0, 1.0, 1.0, 1.0","1.0, 1.0, 1.0, 1.5",,"1.0, 1.0, 1.0, 1.5",...,"1.0, 0.5, 1.0, 4.0","1.0, 1.0, 1.0, 1.5",,,,,,,,
3,2.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,"1.0, 1.0, 1.5, 0.33","1.0, 1.0, 1.0, 1.0","1.0, 1.0, 1.5, 0.33",,"1.0, 1.0, 1.5, 0.33",...,"0.5, 1.0, 4.0, 0.5","1.0, 1.0, 1.5, 0.33",,,,,,,,
4,2.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,,,,,,...,"1.0, 4.0, 0.5, 2.0",,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384,62.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,,,,,,...,"1.0, 0.5, 1.0, 2.0",,,,"1.0, 1.0, 1.0, 1.0",,,,,
385,62.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,,,,,,...,,,,,"1.0, 1.0, 1.0, 2.0",,,,,
386,62.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,,,,,,...,"0.5, 1.0, 2.0, 1.0",,,,"1.0, 1.0, 2.0, 1.0",,,,"0.17, 1.0, 1.0, 1.0",
387,62.0,Missa Morir non può ʼl mio core: Gloria,Valerio Bona,,Bona_2.musicxml,,,,,,...,"1.0, 2.0, 1.0, 4.0",,,,"1.0, 2.0, 1.0, 4.0",,,,"1.0, 1.0, 1.0, 2.0",


## Ricerca normale

In [7]:
output = output.astype(str)

# Assuming 'output' is your DataFrame
search_string = input("Enter the string to search: ")  # Get the search string from the user

# Filtering the DataFrame based on the search string
filtered_df = output[output.apply(lambda row: any(search_string in cell for cell in row), axis=1)]

# Resetting the index
#filtered_df = filtered_df.reset_index(drop=True)

# Drop the column named 'date'
filtered_df = filtered_df.drop(columns=['date', 'composer'])

# Apply styling to the filtered DataFrame using lambda function
styled_df = filtered_df.style.map(lambda val: 'background: #ccebc4' if search_string in str(val) else '')

# Display the styled DataFrame
styled_df

Enter the string to search:  0.33, 2.0, 1.0, 1.0


,Measure,title,path,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,1.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,"0.33, 2.0, 1.0, 1.0",,"0.33, 2.0, 1.0, 0.5","0.33, 2.0, 1.0, 1.0",,,,,,,,
58,11.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,"0.33, 2.0, 1.0, 2.0",,"0.33, 1.0, 0.5, 1.0","0.33, 2.0, 1.0, 2.0",,,"0.33, 2.0, 0.5, 1.0","0.33, 2.0, 1.0, 2.0",,"0.33, 2.0, 1.0, 2.0","0.33, 2.0, 1.0, 2.0","0.33, 2.0, 1.0, 2.0"
82,15.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0","2.0, 2.0, 2.0, 3.0",,,,,,,,,,,,,
132,24.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,,,,,,,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.5",,,,
146,27.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 2.0, 0.5","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5","0.33, 2.0, 2.0, 0.5",,,"0.33, 2.0, 2.0, 0.5","0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5"
148,27.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,,,,,,,,,,,"0.33, 2.0, 1.0, 1.0",
168,30.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,"0.33, 2.0, 0.75, 0.33",,"0.33, 2.0, 4.0, 0.5",,,,,,,,,"0.33, 2.0, 1.0, 1.0",,
232,39.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,,,,,,,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,,
239,40.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 0.5","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,,,,,,,,,,,
248,41.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,"0.33, 2.0, 1.0, 0.5",,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,,,,,,


In [8]:
import io
import contextlib
import sys
import csv

# verovio_canvas_dict
# Define the range dictionary
height_range_dict = {
    range(0, 5): '1200',
    range(5, 10): '2000',
    range(10, 20): '3200'
}

# Function to get the value for a given number
def get_value_for_number(number):
    for key in height_range_dict:
        if number in key:
            return height_range_dict[key]
    return "Number not in any range"

def verovioPrintExampleSave(xml_string, start, stop): # code via github, changed for the save function
    tk = verovio.toolkit()
    tk.loadData(xml_string)
    piece = importScore(xml_string)
    number_voices = len(piece.notes().columns)
    height = get_value_for_number(number_voices)
    tk.setScale(30)
    tk.setOptions({"pageHeight":  height, # Height in pixels
                       "pageWidth":  3000    # Width in pixels
                       })

    if stop == -1:
        meas = piece.measures()
        stop = meas.iloc[-1].tolist()[0]

    mr = str(start) + "-" + str(stop)
    mdict = {'measureRange': mr}

    if stop < start:
        print("Check the measure range, the stop measure must be equal to or greater than the start measure")
    else:
        tk.select(mdict)
        tk.redoLayout()

        # Get the number of pages and save the music as SVG
        count = tk.getPageCount()
        for c in range(1, count + 1):
            music = tk.renderToSVG(c)
            display(HTML(music))
            # file_name = f"{input_file_name}_{measure}_{beat}_{matched}.svg"
            file_name = f"{piece.metadata['title']}_{start}.svg"
            file_path = os.path.join('svg', file_name)
            with open(file_path, "w") as svg_file:
                svg_file.write(music)
                print(f"SVG saved: {file_name}")

        

In [9]:
# NEW FUNCTION
# esegui per generare gli esempi PERSONALIZZATO BONA
for index, row in filtered_df.iterrows():
    # define endpoints
    start = int(float(row['Measure']))
    end = start + 2

    # get xml for verovio
    file_path = row['path']
    with open('Music_Files/'+file_path, 'r') as file:
        xml_string = file.read()

    
        # Call your custom function to print an example
        verovioPrintExampleSave(xml_string, start, end)

SVG saved: Missa Morir non può ʼl mio core: Gloria_1.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_11.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_15.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_24.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_27.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_27.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_30.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_39.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_40.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_41.svg


## Identità salti di 4a e 5a in direzione opposta

In [10]:
import itertools

# Assuming 'output' is your DataFrame
search_string = input("Enter the string to search: ") # Get the search string from the user

def generate_variations(search_string):
    numbers = search_string.split(', ')
    variations = [search_string] # Include the original search string
    for i in range(len(numbers)):
        if numbers[i] == '4':
            numbers[i] = '-5'
        elif numbers[i] == '5':
            numbers[i] = '-4'
        elif numbers[i] == '-4':
            numbers[i] = '5'
        elif numbers[i] == '-5':
            numbers[i] = '4'
        elif numbers[i] == 'P4':
            numbers[i] = '-P5'
        elif numbers[i] == 'P5':
            numbers[i] = '-P4'
        elif numbers[i] == '-P4':
            numbers[i] = 'P5'
        elif numbers[i] == '-P5':
            numbers[i] = 'P4'
        variations.append(', '.join(numbers))
    return variations

# Generate all variations of the search string
variations = generate_variations(search_string)

# Filtering the DataFrame based on the variations
filtered_df_45 = output[output.apply(lambda row: any(any(variation in cell for variation in variations) for cell in row), axis=1)]

# Resetting the index
filtered_df_45 = filtered_df_45.reset_index(drop=True)

# Drop the column named 'date'
filtered_df_45 = filtered_df_45.drop(columns=['date', 'composer'])

# Apply styling to the filtered DataFrame using lambda function
styled_df = filtered_df_45.style.map(lambda val: 'background: #ccebc4' if any(variation in str(val) for variation in variations) else '')

# Display the styled DataFrame
styled_df

Enter the string to search:  0.33, 2.0, 1.0, 1.0


,Measure,title,path,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,1.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,"0.33, 2.0, 1.0, 1.0",,"0.33, 2.0, 1.0, 0.5","0.33, 2.0, 1.0, 1.0",,,,,,,,
1,11.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,"0.33, 2.0, 1.0, 2.0",,"0.33, 1.0, 0.5, 1.0","0.33, 2.0, 1.0, 2.0",,,"0.33, 2.0, 0.5, 1.0","0.33, 2.0, 1.0, 2.0",,"0.33, 2.0, 1.0, 2.0","0.33, 2.0, 1.0, 2.0","0.33, 2.0, 1.0, 2.0"
2,15.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0","2.0, 2.0, 2.0, 3.0",,,,,,,,,,,,,
3,24.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,,,,,,,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.5",,,,
4,27.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 2.0, 0.5","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5","0.33, 2.0, 2.0, 0.5",,,"0.33, 2.0, 2.0, 0.5","0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5",,"0.33, 2.0, 2.0, 0.5"
5,27.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,,,,,,,,,,,"0.33, 2.0, 1.0, 1.0",
6,30.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,"0.33, 2.0, 0.75, 0.33",,"0.33, 2.0, 4.0, 0.5",,,,,,,,,"0.33, 2.0, 1.0, 1.0",,
7,39.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,,,,,,,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,,
8,40.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,"0.33, 2.0, 1.0, 0.5","0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,,,,,,,,,,,
9,41.0,Missa Morir non può ʼl mio core: Gloria,Bona_2.musicxml,,,,,"0.33, 2.0, 1.0, 0.5",,"0.33, 2.0, 1.0, 1.0","0.33, 2.0, 1.0, 1.0",,,,,,,,


In [11]:
# 
for index, row in filtered_df_45.iterrows():
    # define endpoints
    start = int(float(row['Measure']))
    end = start + 2

    # get xml for verovio
    file_path = row['path']
    with open('Music_Files/'+file_path, 'r') as file:
        xml_string = file.read()

    
        # Call your custom function to print an example
        verovioPrintExampleSave(xml_string, start, end)

SVG saved: Missa Morir non può ʼl mio core: Gloria_1.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_11.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_15.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_24.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_27.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_27.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_30.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_39.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_40.svg


SVG saved: Missa Morir non può ʼl mio core: Gloria_41.svg


## Cerca solo nel bc

In [ ]:
# Assuming 'output' is your DataFrame
search_string = input("Enter the string to search: ") # Get the search string from the user

def generate_variations(search_string):
    numbers = search_string.split(', ')
    variations = [search_string] # Include the original search string
    for i in range(len(numbers)):
        if numbers[i] == '4':
            numbers[i] = '-5'
        elif numbers[i] == '5':
            numbers[i] = '-4'
        elif numbers[i] == '-4':
            numbers[i] = '5'
        elif numbers[i] == '-5':
            numbers[i] = '4'
        elif numbers[i] == 'P4':
            numbers[i] = '-P5'
        elif numbers[i] == 'P5':
            numbers[i] = '-P4'
        elif numbers[i] == '-P4':
            numbers[i] = 'P5'
        elif numbers[i] == '-P5':
            numbers[i] = 'P4'
        variations.append(', '.join(numbers))
    return variations

# Generate all variations of the search string
variations = generate_variations(search_string)

# Identify the last non-empty cell in each row
# Replace empty strings with NaN, then forward fill to get the last non-empty value
output_filled = output.replace('', np.nan).ffill()

# Apply the search string variations to the last non-empty cell of each row, ensuring it's not a float
filtered_rows = output_filled.apply(lambda row: any(variation in row.iloc[-1] for variation in variations) if not pd.isna(row.iloc[-1]) and not isinstance(row.iloc[-1], float) else False, axis=1)

# Filter the DataFrame based on the search results
filtered_df_bc = output[filtered_rows]

# Resetting the index
filtered_df_bc = filtered_df_bc.reset_index(drop=True)

# Drop the column named 'date' and 'composer' if they exist
filtered_df_bc = filtered_df_bc.drop(columns=['date', 'composer'], errors='ignore')

# Apply styling to the filtered DataFrame using lambda function
styled_df = filtered_df_bc.style.map(lambda val: 'background: #ccebc4' if any(variation in str(val) for variation in variations) else '')

# Display the styled DataFrame
styled_df

In [ ]:
# Print from the Above DF
for index, row in filtered_df_bc.iterrows():
    # define endpoints
    start = int(float(row['Measure']))
    end = start + 2

    # get xml for verovio
    file_path = row['path']
    with open('Music_Files/'+file_path, 'r') as file:
        xml_string = file.read()

    
        # Call your custom function to print an example
        verovioPrintExampleSave(xml_string, start, end)

In [ ]:
# classifica dei moduli più frequenti

# Exclude the first two columns from the search
columns_to_exclude = new_mel_corpus.columns[:2]
search_columns = [col for col in new_mel_corpus.columns if col not in columns_to_exclude]

# Concatenate all columns to create a series for value counts
all_values = pd.concat([new_mel_corpus[search_columns].apply(lambda row: '|'.join(set(str(val) for val in row if str(val) != 'seguente' and pd.notna(val))), axis=1)])

# Get the most recurring strings
most_recurring_strings = all_values.str.split('|', expand=True).stack().value_counts().reset_index()
most_recurring_strings.columns = ['Module', 'Count']

# Remove rows where 'Module' is an empty cell
most_recurring_strings = most_recurring_strings[most_recurring_strings['Module'] != '']

# Display the resulting DataFrame
most_recurring_strings.head(10)

In [ ]:
filtered_most_recurring = most_recurring_strings[~(
    (most_recurring_strings['Module'].str.count('-2').gt(4) |
     most_recurring_strings['Module'].str.count('2').gt(4) |
     most_recurring_strings['Module'].str.count('1,').gt(4)))].reset_index(drop=True)

# Display the resulting DataFrame
filtered_most_recurring.head(20)

In [ ]:
#writer = pd.ExcelWriter('saved_csv/Riccio_melodic_q_7_cu.xlsx', engine='xlsxwriter')
#new_mel_corpus.to_excel(writer, sheet_name='Sheet1')
#writer.save()

In [ ]:
# Construct the file name with the current date and time
file_name_1 = f'Results.xlsx'

# Write the DataFrame to Excel with the updated file name
filtered_df.to_excel(file_name_1, index=False)

## Versione interattiva

In [ ]:
# ESEGUI LA CELLA e scrivi la tua ricerca
func1 = ImportedPiece.notes
notes_df = corpus.batch(func=func1, kwargs={'combineUnisons': combineUnisons}, metadata=False)
func2 = ImportedPiece.melodic
melodic_df = corpus.batch(func=func2, kwargs={'kind': kind, 'end': False, 'df': notes_df}, metadata=False)
func3 = ImportedPiece.ngrams
ngrams_df = corpus.batch(func=func3, kwargs={'n': n, 'df': melodic_df}, metadata=False)
func4 = ImportedPiece.detailIndex
list_of_detail_index = corpus.batch(func=func4, kwargs={'offset': False,'df': ngrams_df}, metadata=True)

mel_corpus = pd.concat(list_of_detail_index)
comp = mel_corpus.pop("Composer")
mel_corpus['Composer'] = comp
title = mel_corpus.pop("Title")
mel_corpus["Title"] = title
mel_corpus = mel_corpus.fillna('-')

def _convertTuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out

@interact
def mel_ngram_search(my_search="", df = fixed(mel_corpus)):
    df_no_tuple = df.applymap(_convertTuple)
    df_no_tuple.pop("Composer")
    df_no_tuple.pop("Title")
    df_no_tuple.insert(0, "Composer", df["Composer"])
    df_no_tuple.insert(1, "Title", df["Title"])
    filtered_ngrams = df_no_tuple[df_no_tuple.apply(lambda x: x.astype(str).str.contains(my_search).any(), axis=1)].copy()
    
    pd.set_option('max_columns', None)
    return filtered_ngrams.fillna("-").reset_index().applymap(str).style.applymap(lambda x: "background: #ccebc4" if re.search(my_search, x) else "")

### Controlla la musica

In [ ]:
#codice per stampare i titoli nel corpus

In [ ]:
#prefix = 'https://crimproject.org/mei/' 
mei_file = corpus[11] # 0=model 1-5=mass movements (e. g. 2=Gloria)
url = mei_file
piece = importScore(url)
print(piece.metadata)
piece.verovioPrintExample(18, 21) # start measure, end measure

In [ ]:
mei_file = piece_list[13] # 0=model 1-5=mass movements (e. g. 2=Gloria)
url = mei_file
piece = importScore(url)
print(piece.metadata)
piece.verovioPrintExample(36, 41) # start measure, end measure